# Axe 1 — Visualisation des Patterns d'Attention
## Native Sparse Attention — Projet Master IA

---

Ce notebook répond à la question :
> *Pourquoi choisir des blocs plutôt que des tokens individuels ?*

### Plan
1. **Séquence synthétique** : construction d'une séquence à structure blockwise
2. **Matrice d'attention complète** : visualisation de la structure naturellement blockwise
3. **Branche Compression** : scores d'attention sur les tokens compressés
4. **Branche Sélection** : quels blocs sont choisis pour chaque query ?
5. **Branche Sliding Window** : ce qu'elle voit vs ce qu'elle rate
6. **Synthèse** : les 3 branches sont complémentaires


In [ ]:
import os, sys, subprocess

try:
    import google.colab
    IN_COLAB = True
    REPO_URL  = 'https://github.com/YentlCollin/native-sparse-attention.git'
    REPO_NAME = 'native-sparse-attention'

    if not os.path.exists(REPO_NAME):
        subprocess.run(['git', 'clone', '--recursive', REPO_URL], check=True)
    else:
        subprocess.run(['git', '-C', REPO_NAME, 'pull'], check=True)
        subprocess.run(['git', '-C', REPO_NAME, 'submodule', 'update', '--init', '--recursive'], check=True)

    if os.path.basename(os.getcwd()) != REPO_NAME:
        os.chdir(REPO_NAME)

    subprocess.run([sys.executable, '-m', 'pip', 'install', 'ninja', 'packaging', 'einops', '-q'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '-q', '--no-build-isolation'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'matplotlib', 'seaborn', '-q'], check=True)
    print('Setup Colab OK ✓')
except ImportError:
    IN_COLAB = False
    print('Local')

os.makedirs('notebooks/figures', exist_ok=True)
FIGURE_DIR = 'notebooks/figures'


In [ ]:
import math
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from native_sparse_attention.ops.naive import compression

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float32   # float32 pour la précision des scores

print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

plt.rcParams.update({'font.size': 12, 'axes.titlesize': 13, 'figure.dpi': 120})

# ── Hyperparamètres NSA (identiques au papier) ─────────────────────
H_Q        = 16    # Query heads (réduit pour la visu)
H          = 1     # KV heads (GQA ratio=16)
D          = 64    # Head dimension
BLOCK_SIZE = 64    # Taille d'un bloc
BLOCK_COUNTS = 4   # Blocs sélectionnés (réduit pour la visu)
WINDOW_SIZE  = 64  # Sliding window
B = 1

print(f'Config : H_Q={H_Q}, H={H}, D={D}, block={BLOCK_SIZE}, S={BLOCK_COUNTS}, win={WINDOW_SIZE}')


---
## Section 1 — Séquence Synthétique à Structure Blockwise

**Hypothèse clé du papier (Figure 8) :**
> *Les scores d'attention montrent une continuité spatiale — les tokens voisins ont des scores similaires.*

Pour le démontrer, on crée une séquence où chaque bloc a une identité distincte :
- Les tokens d'un même bloc partagent le même embedding de base
- Les blocs sont orthogonaux entre eux
- Les queries sont alignées avec des blocs spécifiques


In [ ]:
torch.manual_seed(42)

T        = 512           # Longueur de séquence
n_blocks = T // BLOCK_SIZE  # 8 blocs

# ── Création de la séquence structurée ────────────────────────────
# Chaque bloc a un embedding de base distinct + bruit faible
block_bases = F.normalize(torch.randn(n_blocks, D), dim=-1)  # [8, D] orthogonaux

k = torch.zeros(B, T, H, D, dtype=dtype)
v = torch.zeros(B, T, H, D, dtype=dtype)

for b in range(n_blocks):
    start, end = b * BLOCK_SIZE, (b + 1) * BLOCK_SIZE
    noise = 0.1 * torch.randn(BLOCK_SIZE, H, D)
    k[0, start:end] = block_bases[b] + noise  # tokens similaires dans chaque bloc
    v[0, start:end] = torch.randn(BLOCK_SIZE, H, D)

# Queries : chaque query cherche le bloc auquel elle appartient
q = torch.zeros(B, T, H_Q, D, dtype=dtype)
for b in range(n_blocks):
    start, end = b * BLOCK_SIZE, (b + 1) * BLOCK_SIZE
    noise = 0.2 * torch.randn(BLOCK_SIZE, H_Q, D)
    q[0, start:end] = block_bases[b] + noise

# Query 'probe' : cherche le bloc 0 depuis la fin de la séquence
PROBE_POS   = T - 1       # position de la query de test
TARGET_BLOCK = 0          # bloc cible (le plus ancien)
q[0, PROBE_POS] = block_bases[TARGET_BLOCK].unsqueeze(0).expand(H_Q, -1) + \
                  0.1 * torch.randn(H_Q, D)

k, v, q = k.to(device), v.to(device), q.to(device)

print(f'Séquence : T={T}, n_blocs={n_blocks}')
print(f'Query probe à t={PROBE_POS} → cible : bloc {TARGET_BLOCK} (tokens 0-{BLOCK_SIZE-1})')

# Vérification : similarité cosinus entre bloc_bases
sim = block_bases @ block_bases.T
print(f'Similarité max hors diag entre blocs : {sim.fill_diagonal_(0).abs().max().item():.3f}')
print('(proche de 0 = blocs bien séparés)')


---
## Section 2 — Matrice d'Attention Complète

On calcule la **matrice d'attention causale complète** pour visualiser
sa structure naturellement blockwise.

C'est ce que l'attention dense calcule — $O(T^2)$ opérations.


In [ ]:
scale = D ** -0.5
G = H_Q // H

# Expand k to H_Q for full attention
k_exp = k.repeat_interleave(G, dim=2)  # [B, T, H_Q, D]

# Full causal attention matrix [H_Q, T, T]
# On utilise la première tête pour la visualisation
with torch.no_grad():
    scores = torch.einsum('bthd,bshd->bhts',
                          q.float(), k_exp.float()) * scale  # [B, H_Q, T, T]
    causal_mask = torch.triu(torch.ones(T, T, device=device, dtype=torch.bool), diagonal=1)
    scores = scores.masked_fill(causal_mask[None, None], float('-inf'))
    attn_full = scores.softmax(-1)  # [B, H_Q, T, T]

attn_avg = attn_full[0].mean(0).cpu().numpy()  # [T, T] moyenne sur les têtes

# ── Figure : Matrice d'attention ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Matrice d\'Attention Causale Complète\n'
             '(Séquence synthétique structurée — 8 blocs de 64 tokens)',
             fontsize=14, fontweight='bold')

# Heatmap complète
ax = axes[0]
im = ax.imshow(attn_avg, cmap='Blues', aspect='auto', origin='upper',
               vmin=0, vmax=attn_avg.max())
plt.colorbar(im, ax=ax, label='Score d\'attention')
# Grilles de blocs
for b in range(1, n_blocks):
    ax.axvline(b * BLOCK_SIZE - 0.5, color='red', lw=0.8, alpha=0.6)
    ax.axhline(b * BLOCK_SIZE - 0.5, color='red', lw=0.8, alpha=0.6)
ax.set_xlabel('Position Key (source)')
ax.set_ylabel('Position Query (target)')
ax.set_title('Matrice complète\n(grilles rouges = limites de blocs)')
# Annotations blocs
for b in range(n_blocks):
    mid = b * BLOCK_SIZE + BLOCK_SIZE // 2
    ax.text(mid, -15, f'B{b}', ha='center', va='top', fontsize=7, color='darkred')

# Zoom sur la query probe
ax = axes[1]
probe_attn = attn_full[0, :, PROBE_POS, :].mean(0).cpu().numpy()  # [T]
ax.bar(range(T), probe_attn, color='steelblue', alpha=0.7, width=1.0)
for b in range(n_blocks):
    ax.axvspan(b * BLOCK_SIZE, (b+1) * BLOCK_SIZE - 1,
               alpha=0.05, color='red' if b == TARGET_BLOCK else 'gray')
    ax.text(b * BLOCK_SIZE + BLOCK_SIZE//2, max(probe_attn)*0.95,
            f'B{b}', ha='center', fontsize=8, color='darkred' if b == TARGET_BLOCK else 'gray')
ax.set_xlabel('Position (token)')
ax.set_ylabel('Score d\'attention')
ax.set_title(f'Query probe (t={PROBE_POS}) → distribution des scores\n'
             f'Bloc cible = B{TARGET_BLOCK} (rouge)')

plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/axe1_full_attention.png', bbox_inches='tight', dpi=150)
plt.show()
print('\nObservation : les scores sont concentrés PAR BLOCS → structure blockwise naturelle.')
print('C\'est ce qui motive la sélection par blocs dans NSA.')


---
## Section 3 — Branche Compression

La branche compression **réduit chaque bloc de $l=64$ tokens en 1 token**
via mean pooling :

$$\tilde{k}_i = \frac{1}{l} \sum_{j=il}^{(i+1)l-1} k_j$$

Résultat : une séquence de $T/64 = 8$ tokens compressés capturant le contexte **global**.

Le score d'attention $q_t \cdot \tilde{k}_i$ permet de **classer les blocs par importance**
pour chaque query → c'est le signal utilisé pour la sélection.


In [ ]:
with torch.no_grad():
    # Mean pooling des blocs
    k_cmp, v_cmp = compression(k, v, BLOCK_SIZE)  # [B, C, H, D]
    C = k_cmp.shape[1]  # nombre de blocs compressés

    # Expand to H_Q
    k_cmp_exp = k_cmp.repeat_interleave(G, dim=2)  # [B, C, H_Q, D]

    # Scores d'attention entre queries et blocs compressés
    scores_cmp = torch.einsum('bthd,bchd->bhtc',
                               q.float(), k_cmp_exp.float()) * scale  # [B, H_Q, T, C]

    # Masque causal : bloc i visible pour query à t si i*block_size <= t
    t_idx = torch.arange(T, device=device)
    c_idx = torch.arange(C, device=device) * BLOCK_SIZE
    causal_cmp = c_idx[None, :] > t_idx[:, None]  # [T, C]
    scores_cmp = scores_cmp.masked_fill(causal_cmp[None, None], float('-inf'))
    attn_cmp = scores_cmp.softmax(-1)  # [B, H_Q, T, C]

    # Top-k block selection
    attn_cmp_avg = attn_cmp[0].mean(0)  # [T, C] moyenne têtes
    topk_indices = attn_cmp_avg.topk(BLOCK_COUNTS, dim=-1).indices  # [T, S]

attn_cmp_np   = attn_cmp_avg.cpu().numpy()
topk_np       = topk_indices.cpu().numpy()

# ── Figure ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Branche Compression\n'
             f'T={T} tokens → C={C} tokens compressés (ratio ×{T//C})',
             fontsize=14, fontweight='bold')

# Heatmap scores compression [T × C]
ax = axes[0]
im = ax.imshow(attn_cmp_np, cmap='YlOrRd', aspect='auto', origin='upper')
plt.colorbar(im, ax=ax, label='Score d\'attention (softmax)')
ax.set_xlabel('Bloc compressé (index)')
ax.set_ylabel('Position Query')
ax.set_title('Scores d\'attention Query → Blocs compressés\n(conscience globale)')
ax.set_xticks(range(C))
ax.set_xticklabels([f'B{i}' for i in range(C)])
# Marquer la diagonale (bloc local)
diag_x = np.arange(C)
diag_y = (diag_x + 0.5) * BLOCK_SIZE
ax.plot(diag_x, diag_y, 'b--', lw=1.5, label='Bloc local (diagonale)', alpha=0.7)
ax.legend(fontsize=9)

# Scores pour la query probe
ax = axes[1]
probe_scores = attn_cmp_np[PROBE_POS]  # [C]
colors = ['#e74c3c' if i == TARGET_BLOCK else
          '#2ecc71' if i in topk_np[PROBE_POS] else '#95a5a6'
          for i in range(C)]
bars = ax.bar(range(C), probe_scores, color=colors, edgecolor='white', lw=1.5)
ax.set_xticks(range(C))
ax.set_xticklabels([f'B{i}' for i in range(C)])
ax.set_xlabel('Bloc compressé')
ax.set_ylabel('Score d\'attention')
ax.set_title(f'Query probe (t={PROBE_POS}) → scores par bloc')
legend_els = [
    mpatches.Patch(color='#e74c3c', label=f'Bloc cible (B{TARGET_BLOCK})'),
    mpatches.Patch(color='#2ecc71', label=f'Top-{BLOCK_COUNTS} sélectionnés'),
    mpatches.Patch(color='#95a5a6', label='Non sélectionnés'),
]
ax.legend(handles=legend_els, fontsize=9)

# Annotations valeurs
for bar, val in zip(bars, probe_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/axe1_compression.png', bbox_inches='tight', dpi=150)
plt.show()

target_in_topk = TARGET_BLOCK in topk_np[PROBE_POS]
print(f'Bloc cible B{TARGET_BLOCK} dans le top-{BLOCK_COUNTS} : {target_in_topk} ✓' if target_in_topk
      else f'Bloc cible B{TARGET_BLOCK} dans le top-{BLOCK_COUNTS} : NON ✗')


---
## Section 4 — Branche Sélection

Les scores de compression déterminent quels **blocs complets** sont
récupérés pour l'attention fine :

$$\mathcal{I}_t = \{i \mid \text{rank}(\tilde{p}_t[i]) \leq S\}$$

Visualisation : pour chaque position de query $t$, quels blocs sont sélectionnés ?


In [ ]:
# Matrice de sélection : 1 si le bloc b est sélectionné pour la query t
selection_matrix = np.zeros((T, C))
for t in range(T):
    for b in topk_np[t]:
        if 0 <= b < C:
            selection_matrix[t, b] = 1.0

# ── Figure ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Branche Sélection\n'
             f'Top-{BLOCK_COUNTS} blocs sélectionnés par query (sur {C} blocs disponibles)',
             fontsize=14, fontweight='bold')

# Heatmap sélection
ax = axes[0]
ax.imshow(selection_matrix, cmap='Greens', aspect='auto', origin='upper',
          vmin=0, vmax=1)
ax.set_xlabel('Bloc sélectionné (index)')
ax.set_ylabel('Position Query')
ax.set_title(f'Matrice de Sélection\n(vert = bloc sélectionné, S={BLOCK_COUNTS}/{C})')
ax.set_xticks(range(C))
ax.set_xticklabels([f'B{i}' for i in range(C)])
# Sliding window coverage
win_blocks = WINDOW_SIZE // BLOCK_SIZE
for t in range(0, T, BLOCK_SIZE):
    b_start = max(0, t // BLOCK_SIZE - win_blocks)
    b_end   = t // BLOCK_SIZE
    if b_start < b_end:
        ax.plot([b_start - 0.5, b_end - 0.5], [t, t], 'b-', lw=0.5, alpha=0.3)

# Fréquence de sélection par bloc
ax = axes[1]
freq = selection_matrix.mean(0)  # [C]
colors = ['#e74c3c' if i == TARGET_BLOCK else '#2ecc71' for i in range(C)]
ax.bar(range(C), freq, color=colors, edgecolor='white', lw=1.5)
ax.axhline(BLOCK_COUNTS / C, color='gray', ls='--', lw=1.5,
           label=f'Sélection uniforme ({BLOCK_COUNTS}/{C}={BLOCK_COUNTS/C:.2f})')
ax.set_xticks(range(C))
ax.set_xticklabels([f'B{i}' for i in range(C)])
ax.set_xlabel('Bloc')
ax.set_ylabel('Fréquence de sélection')
ax.set_title('Fréquence de Sélection par Bloc\n(tous les queries confondus)')
ax.legend(fontsize=9)
for i, val in enumerate(freq):
    ax.text(i, val + 0.01, f'{val:.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/axe1_selection.png', bbox_inches='tight', dpi=150)
plt.show()
print('Observation : le bloc 0 (le plus ancien) est sélectionné plus souvent car')
print('les queries de fin de séquence lui ressemblent (structure synthétique).')


---
## Section 5 — Branche Sliding Window vs NSA

La sliding window ne voit que les `window_size` derniers tokens.
Pour les long-range dependencies, elle est aveugle.


In [ ]:
# ── Visualisation : ce que voit chaque méthode pour la query probe ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Comparaison des 3 branches — Query probe à t={PROBE_POS}\n'
             f'Cible : Bloc B{TARGET_BLOCK} (tokens 0-{BLOCK_SIZE-1})',
             fontsize=14, fontweight='bold')

# Vision de chaque méthode
visible_dense  = np.ones(T)   # voit tout (attention causale)
visible_window = np.zeros(T)  # seulement les W derniers tokens
visible_window[max(0, PROBE_POS - WINDOW_SIZE + 1): PROBE_POS + 1] = 1.0

visible_nsa = np.zeros(T)
# Sélection NSA
for b in topk_np[PROBE_POS]:
    if 0 <= b < C:
        start = b * BLOCK_SIZE
        end   = min((b + 1) * BLOCK_SIZE, T)
        visible_nsa[start:end] = 0.7
# Window dans NSA
visible_nsa[max(0, PROBE_POS - WINDOW_SIZE + 1): PROBE_POS + 1] = 1.0

# Score d'attention pour les tokens visibles par NSA
probe_full = attn_full[0, :, PROBE_POS, :].mean(0).cpu().numpy()  # [T]

for ax, visible, title, color in [
    (axes[0], visible_window, f'Sliding Window\n(window={WINDOW_SIZE})', '#3498db'),
    (axes[1], visible_nsa,    f'NSA\n(S={BLOCK_COUNTS} blocs + window={WINDOW_SIZE})', '#2ecc71'),
    (axes[2], probe_full,     'Attention Dense\n(référence)', '#e74c3c'),
]:
    ax.bar(range(T), visible, color=color, alpha=0.6, width=1.0)
    # Zone cible
    ax.axvspan(TARGET_BLOCK * BLOCK_SIZE, (TARGET_BLOCK+1) * BLOCK_SIZE,
               alpha=0.25, color='gold', label=f'Bloc cible B{TARGET_BLOCK}')
    ax.set_xlabel('Position token')
    ax.set_ylabel('Score / visibilité')
    ax.set_title(title)
    ax.legend(fontsize=9)

# Résumé textuel
needle_visible_win = visible_window[:BLOCK_SIZE].sum() > 0
needle_visible_nsa = visible_nsa[:BLOCK_SIZE].sum() > 0

plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/axe1_branches_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'Bloc cible visible par Sliding Window : {needle_visible_win}')
print(f'Bloc cible visible par NSA            : {needle_visible_nsa}')
print(f'\nConclusion : NSA capture le bloc cible B{TARGET_BLOCK} à distance {PROBE_POS} tokens')
print(f'grâce à la branche compression. La sliding window est aveugle à cette distance.')


---
## Section 6 — Synthèse : Les 3 branches sont complémentaires


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Synthèse — Architecture NSA : 3 branches complémentaires\n'
             'Séquence synthétique T=512, 8 blocs de 64 tokens',
             fontsize=15, fontweight='bold')

# [0,0] Matrice attention complète (zoom 128 premiers tokens)
ax = axes[0, 0]
Z = 128
im = ax.imshow(attn_avg[:Z, :Z], cmap='Blues', aspect='auto', origin='upper')
for b in range(1, Z // BLOCK_SIZE):
    ax.axvline(b*BLOCK_SIZE - 0.5, color='red', lw=1, alpha=0.7)
    ax.axhline(b*BLOCK_SIZE - 0.5, color='red', lw=1, alpha=0.7)
plt.colorbar(im, ax=ax)
ax.set_title('Attention Dense — Structure Blockwise\n(zoom 128 tokens, grilles = blocs)')
ax.set_xlabel('Key position'); ax.set_ylabel('Query position')

# [0,1] Scores compression
ax = axes[0, 1]
im = ax.imshow(attn_cmp_np, cmap='YlOrRd', aspect='auto', origin='upper')
plt.colorbar(im, ax=ax, label='Score')
ax.set_title(f'Branche Compression — Conscience Globale\n({C} blocs compressés)')
ax.set_xlabel('Bloc compressé'); ax.set_ylabel('Query position')
ax.set_xticks(range(C)); ax.set_xticklabels([f'B{i}' for i in range(C)])

# [1,0] Sélection
ax = axes[1, 0]
ax.imshow(selection_matrix, cmap='Greens', aspect='auto', origin='upper', vmin=0, vmax=1)
ax.set_title(f'Branche Sélection — Top-{BLOCK_COUNTS} Blocs par Query\n(granularité fine)')
ax.set_xlabel('Bloc sélectionné'); ax.set_ylabel('Query position')
ax.set_xticks(range(C)); ax.set_xticklabels([f'B{i}' for i in range(C)])

# [1,1] Comparaison couverture
ax = axes[1, 1]
methods = ['Dense\n(O(T²))', 'NSA\n(O(T·k))', 'Sliding Window\nseul O(T·w)']
coverage = [
    100.0,
    (BLOCK_COUNTS * BLOCK_SIZE + WINDOW_SIZE) / T * 100,
    WINDOW_SIZE / T * 100,
]
colors = ['#e74c3c', '#2ecc71', '#3498db']
bars = ax.bar(methods, coverage, color=colors, edgecolor='white', lw=2)
ax.set_ylabel('% tokens accédés par query')
ax.set_title('Couverture par Méthode\n(tokens KV accédés / T)')
for bar, val in zip(bars, coverage):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/axe1_synthesis.png', bbox_inches='tight', dpi=150)
plt.show()

print('Résumé :')
print(f'  Dense         : accède à 100% des tokens (O(T²))')
print(f'  NSA           : accède à ~{coverage[1]:.1f}% des tokens (O(T·k))')
print(f'  Sliding Window: accède à ~{coverage[2]:.1f}% des tokens (O(T·w))')
print(f'\nNSA = meilleur compromis : global (compression) + précis (sélection) + local (window)')
